# CareLoom: Multi-Agent Onboarding Briefs — Larkspur Electronics

Larkspur's support lead has a new hire starting Monday, and wants a clean, well-sourced
briefing on the refund and warranty policy before they take their first ticket — the same
policy handbook the notebook before this one made queryable. Writing that briefing well is
really three jobs: pull the relevant evidence, draft it clearly, and check it doesn't
overstate anything the handbook doesn't actually say. This notebook builds that as a real
multi-agent system: three Groq-backed subagents coordinated by a supervisor.

Use case:
- Researcher agent pulls evidence from Larkspur's real policy handbook
- Writer agent drafts an onboarding brief
- QA agent checks the draft for unsupported claims and clarity
- Supervisor node coordinates the workflow

LangChain's multi-agent docs say a main agent coordinates subagents as tools, and that subagents are useful when you need centralized control, parallelization, or specialized knowledge. The component-architecture page also lists multi-agent systems as a core LangChain pattern.

## Why this use case

An onboarding brief is a good fit for a multi-agent graph because the work naturally splits into specialized steps:

- research evidence from the handbook
- draft a readable brief
- verify that the brief stays grounded in the source

This matches the subagent pattern where the main orchestrator maintains the conversation while subagents stay specialized.

## Learning goals

By the end of this notebook, you should be able to:

1. Wire real Groq-backed subagents into the supervisor/worker shape from the earlier multi-agent notebook.
2. Give a Researcher agent a retrieval tool grounded in a real document.
3. Coordinate Researcher, Writer, and QA agents with a supervisor loop.
4. Keep every agent call safe to run even without live model access.


## 1) Install packages


In [ ]:
%pip install -qU \
    python-dotenv fpdf2 pypdf \
    langchain langchain-core langchain-groq \
    langchain-chroma chromadb langchain-huggingface sentence-transformers \
    langgraph numpy


## 2) Load environment variables

Create a `.env` file with:

```env
GROQ_API_KEY=your_groq_api_key
LANGSMITH_API_KEY=your_langsmith_api_key
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=careloom-onboarding-brief
```


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
LANGSMITH_API_KEY = os.getenv('LANGSMITH_API_KEY')
LANGSMITH_TRACING = os.getenv('LANGSMITH_TRACING', 'true').lower() == 'true'
LANGSMITH_PROJECT = os.getenv('LANGSMITH_PROJECT', 'careloom-onboarding-brief')

print('GROQ_API_KEY set:', bool(GROQ_API_KEY))
print('LANGSMITH_API_KEY set:', bool(LANGSMITH_API_KEY))
print('LANGSMITH_TRACING:', LANGSMITH_TRACING)
print('LANGSMITH_PROJECT:', LANGSMITH_PROJECT)


## 3) Prepare the reference document

Same idea as the previous notebook: instead of downloading a public PDF that has nothing to do with Larkspur, we generate the real document the Researcher agent will actually cite from — Larkspur's own Customer Care & Policy Handbook — as a genuine multi-page PDF, with no network dependency. This notebook is self-contained, so it regenerates the same handbook rather than importing it from the previous one.


In [ ]:
from pathlib import Path
from fpdf import FPDF, XPos, YPos

DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

PDF_PATH = DATA_DIR / "larkspur_care_policy_handbook.pdf"

HANDBOOK_SECTIONS = [
    ("Welcome", (
        "This handbook is the shared reference for everyone on Larkspur Electronics' "
        "customer care team, whether you are a new hire in your first week or a support "
        "lead reviewing an escalation. It covers what customers are entitled to, how long "
        "each process takes, and where to draw the line between handling something "
        "yourself and looping in a supervisor. When a policy in this handbook conflicts "
        "with what a customer says a previous agent told them, this handbook is the "
        "source of truth; note the discrepancy in the ticket and flag it to your team "
        "lead so the record can be corrected.\n\n"
        "Larkspur sells keyboards, mice, headphones, monitors, webcams, and desk lighting "
        "directly to consumers across India, with a smaller volume of international "
        "orders. Most of what follows applies to both, with shipping timelines being the "
        "main exception."
    )),
    ("Returns Policy", (
        "Customers may return an unopened item within 14 days of delivery for a full "
        "refund. The item must be in its original packaging with all accessories, "
        "manuals, and included cables. A restocking fee does not apply to unopened "
        "returns.\n\n"
        "Opened items may be returned within 7 days of delivery only if the product is "
        "defective or was not as described in the listing. Not as described covers wrong "
        "color, wrong model, or missing advertised features -- it does not cover a "
        "customer simply changing their mind after opening the box. Opened, "
        "non-defective returns are not accepted; direct the customer to our exchange "
        "program instead if they want a different model.\n\n"
        "Items marked as final sale (typically clearance and open-box listings) cannot "
        "be returned under any circumstances except a manufacturing defect confirmed by "
        "our technical team. This exception always applies regardless of the final-sale "
        "label, since a defective product doesn't stop being defective because it was "
        "discounted.\n\n"
        "To start a return, the customer needs the order number and the email address "
        "used at checkout. Return shipping is free for defective items; for a "
        "change-of-mind return on an unopened item, the customer covers return shipping "
        "unless the order qualified for free returns at checkout (orders above Rs. "
        "5,000 currently qualify)."
    )),
    ("Refunds Policy", (
        "Refunds are issued within 7 business days of the returned item passing "
        "inspection at our warehouse -- not from the day the customer ships it back. "
        "Inspection typically takes 1-2 business days after the item arrives, so "
        "customers should expect roughly 8-9 business days total from drop-off to "
        "refund.\n\n"
        "Refunds are issued to the original payment method whenever possible. If the "
        "original payment method is no longer valid (an expired card, for example), the "
        "refund is issued as store credit unless the customer specifically requests a "
        "bank transfer, which takes an additional 3-5 business days to process.\n\n"
        "Refunds above Rs. 3,000 must be approved by a support lead before they are "
        "processed. This is not a reflection on the customer or the claim -- it is a "
        "standing control on any outbound payment past that threshold, and it applies "
        "the same way whether the request comes from a first-time customer or someone "
        "who has ordered from us for years. Flag the ticket for approval and let the "
        "customer know a supervisor is reviewing it; most approvals are same-day.\n\n"
        "Partial refunds are available when only part of an order is being returned, or "
        "when a customer keeps a damaged item rather than shipping it back (common for "
        "large monitors, where return shipping is expensive and slow). A partial refund "
        "in that case is typically 50-70% of the item's price, depending on the extent "
        "of the damage, and always requires supervisor approval regardless of amount."
    )),
    ("Warranty Policy", (
        "Every product Larkspur sells carries a 1-year manufacturer's warranty against "
        "defects in materials and workmanship, starting from the delivery date. This "
        "covers things like a keyboard with keys that stop registering, a mouse sensor "
        "that drifts, headphones with a dead channel, or a monitor with stuck pixels "
        "beyond the panel manufacturer's acceptable threshold.\n\n"
        "The warranty does not cover physical damage, liquid damage, damage from "
        "unauthorized repair attempts, or normal cosmetic wear such as fading keycap "
        "labels. If a customer describes dropping the product, spilling a drink on it, "
        "or opening the case themselves, that claim falls outside warranty coverage -- "
        "explain this clearly and offer a paid repair estimate instead if one is "
        "available.\n\n"
        "To file a warranty claim, the customer needs proof of purchase and a short "
        "description or photo of the issue. Our technical team reviews claims within 3 "
        "business days. Approved claims are resolved as a replacement unit, a repair, "
        "or -- if neither is available in reasonable time -- a refund at the support "
        "lead's discretion. Replacement is the default remedy; only escalate to a refund "
        "conversation if a replacement unit is genuinely unavailable within 30 days."
    )),
    ("Shipping & Delivery", (
        "Standard delivery within Chennai, Bengaluru, Mumbai, Delhi, and Pune metro "
        "limits takes 3-5 business days. Delivery outside these metro areas typically "
        "takes 5-8 business days. Expedited shipping, where available, cuts this to 1-3 "
        "business days for an additional fee shown at checkout.\n\n"
        "During major sale periods and the festival season (roughly mid-October through "
        "early November, and again in late December), delivery estimates should be "
        "extended by 2-3 additional business days across all regions due to courier "
        "network volume. Proactively mention this to customers ordering during these "
        "windows rather than waiting for them to ask why a package is running late.\n\n"
        "Every order includes tracking, sent by email at the time of dispatch. If a "
        "customer says tracking hasn't updated in more than 3 business days, check the "
        "carrier's status directly rather than only re-sending the same tracking link "
        "-- carrier-side delays are more common during festival season and courier "
        "strikes, and customers appreciate a real update over a repeated link.\n\n"
        "Orders that are lost in transit (no tracking movement for 10+ business days, "
        "confirmed with the carrier) are handled as a full replacement shipment at no "
        "cost to the customer, without needing supervisor approval -- this is a "
        "standing exception separate from the general refund-approval threshold, since "
        "the customer paid for a delivery that didn't happen."
    )),
    ("Escalation Matrix", (
        "Most tickets should be resolved by a Tier 1 agent within one interaction, using "
        "this handbook. Escalate to Tier 2 when a case needs technical judgment beyond "
        "this handbook -- for example, whether a described issue is covered under "
        "warranty when it's genuinely ambiguous.\n\n"
        "Escalate directly to a support lead (skipping Tier 2) for: any refund above Rs. "
        "3,000, any partial refund of any amount, a customer requesting a manager by "
        "name, or a customer who has contacted us three or more times about the same "
        "unresolved issue.\n\n"
        "Response-time targets: Tier 1 first response within 4 business hours; Tier 2 "
        "within 1 business day of escalation; support-lead review of a pending refund "
        "approval within the same business day it's flagged. These are targets, not "
        "guarantees, but a ticket approaching double its target time should be flagged "
        "in the daily stand-up regardless of who owns it."
    )),
]


def build_handbook_pdf(path: Path) -> Path:
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=18)
    pdf.add_page()

    pdf.set_font("Helvetica", "B", 18)
    pdf.multi_cell(0, 10, "Larkspur Electronics", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.set_font("Helvetica", "", 13)
    pdf.multi_cell(0, 8, "Customer Care & Policy Handbook", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.ln(4)

    for heading, body in HANDBOOK_SECTIONS:
        pdf.set_font("Helvetica", "B", 14)
        pdf.multi_cell(0, 8, heading, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font("Helvetica", "", 11)
        pdf.multi_cell(0, 6, body, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.ln(4)

    pdf.output(str(path))
    return path


build_handbook_pdf(PDF_PATH)
print("Handbook PDF ready:", PDF_PATH.resolve(), f"({PDF_PATH.stat().st_size} bytes)")


## 4) Load and chunk the PDF

LangChain's knowledge-base flow is load → split → embed → store. We use that pattern here for the Researcher agent.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(pages)

print("Pages:", len(pages))
print("Chunks:", len(chunks))
print("First page preview:")
print(pages[0].page_content[:500])


## 5) Create a ChromaDB knowledge store

Chroma is the semantic memory for the document. We store handbook chunks here so the Researcher agent can retrieve supporting evidence by meaning.

Same offline-safety pattern as the previous notebook: try the real `sentence-transformers` embeddings first, fall back to a dependency-free hashing embedding if that model can't be reached. This notebook uses its own persist directory and collection name, separate from the previous notebook's, and always clears it before rebuilding — so running either notebook, in either order, never mixes stale vectors from a different run into the same collection.


In [ ]:
import shutil
import hashlib
from pathlib import Path
import numpy as np
import chromadb
from langchain_core.embeddings import Embeddings
from langchain_chroma import Chroma


class LocalHashEmbeddings(Embeddings):
    """
    Deterministic, dependency-free fallback embedding function. Used only when
    the real sentence-transformers model can't be reached. It hashes each
    token into a fixed-size bag-of-words vector -- mechanically sound for
    demonstrating similarity search, but semantically much weaker than a real
    sentence embedding (no sense of synonyms or word order).
    """

    def __init__(self, dim: int = 384):
        self.dim = dim

    def _embed(self, text: str) -> list[float]:
        vec = np.zeros(self.dim)
        for token in text.lower().split():
            h = int(hashlib.sha256(token.encode("utf-8")).hexdigest(), 16)
            vec[h % self.dim] += 1.0
        norm = np.linalg.norm(vec)
        return (vec / norm).tolist() if norm > 0 else vec.tolist()

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed(t) for t in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embed(text)


def build_embeddings():
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
        emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        emb.embed_query("connectivity check")
        print("Using live HuggingFace embeddings (sentence-transformers/all-MiniLM-L6-v2).")
        return emb
    except Exception as e:
        print(f"[offline-fallback] HuggingFace embeddings unavailable ({type(e).__name__}: {e}).")
        print("[offline-fallback] Using LocalHashEmbeddings instead.")
        return LocalHashEmbeddings()


embeddings = build_embeddings()

CHROMA_DIR = Path("./data/chroma_brief_store")
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

vector_store = Chroma(
    client=client,
    collection_name="larkspur_handbook_brief",
    embedding_function=embeddings,
)

vector_store.add_documents(chunks)

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print("Chroma store ready at", CHROMA_DIR.resolve())


## 6) Create the Groq model

The supervisor and workers use Groq as the model provider. You can change the model name with `GROQ_MODEL` in `.env` if needed.


In [ ]:
from langchain_groq import ChatGroq

GROQ_MODEL = os.getenv('GROQ_MODEL', 'openai/gpt-oss-120b')
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

print('Groq model:', GROQ_MODEL)


## 7) Build three specialist subagents

The subagents docs show the core idea of wrapping specialized work behind agents that the supervisor can coordinate. Subagents are stateless and the main orchestrator keeps the overall conversation context.


In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def retrieve_evidence(query: str) -> str:
    'Retrieve the most relevant handbook chunks for a question.'
    docs = retriever.invoke(query)
    lines = []
    for doc in docs:
        page = doc.metadata.get('page', 'n/a')
        snippet = doc.page_content[:700].replace('\n', ' ')
        lines.append(f'[page={page}] {snippet}')
    return '\n\n'.join(lines)

research_agent = create_agent(
    model=llm,
    tools=[retrieve_evidence],
    system_prompt=(
        'You are the Researcher agent. Use the retrieve_evidence tool to gather supporting facts '
        "from Larkspur Electronics' Customer Care & Policy Handbook. "
        'Return 3 to 5 concise evidence bullets and include page numbers when possible.'
    ),
)

writer_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        'You are the Writer agent. Turn the research notes into a concise onboarding brief with headings. '
        'Use only the evidence provided to you.'
    ),
)

qa_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        'You are the QA agent. Check the brief for unsupported claims, missing evidence, and clarity issues. '
        'Return a short QA report and a revised final brief.'
    ),
)

print('Researcher, Writer, and QA agents are ready.')


## 8) Define the graph state

The supervisor graph keeps the question, the research notes, the draft, the QA report, and the final brief.


In [ ]:
import operator
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class SupervisorState(TypedDict):
    messages: Annotated[list, operator.add]
    question: str
    stage: str
    research_notes: str
    draft: str
    qa_report: str
    final_brief: str


## 9) Supervisor logic

The supervisor decides which stage should run next.

This follows the multi-agent pattern where a central main agent coordinates subagents as tools and keeps control over the workflow.


In [ ]:
def supervisor_node(state: SupervisorState):
    if not state.get('research_notes'):
        next_stage = 'research'
    elif not state.get('draft'):
        next_stage = 'write'
    elif not state.get('qa_report'):
        next_stage = 'qa'
    else:
        next_stage = 'finish'
    return {'stage': next_stage}

def route_from_supervisor(state: SupervisorState) -> Literal['research', 'write', 'qa', 'finish']:
    return state['stage']


## 10) Worker nodes

Each worker calls its specialist subagent and stores the result in the shared graph state. `safe_agent_invoke` wraps that call so a missing key or unreachable Groq endpoint degrades to a clearly-labeled placeholder instead of stopping the whole pipeline partway through.


In [ ]:
def get_last_message_text(result: dict) -> str:
    msgs = result.get('messages', [])
    for msg in reversed(msgs):
        content = getattr(msg, 'content', None)
        if content:
            return content
        if isinstance(msg, dict):
            if msg.get('content'):
                return msg['content']
    return ''


def safe_agent_invoke(agent, prompt: str, fallback_text: str) -> str:
    try:
        result = agent.invoke({'messages': [('user', prompt)]})
        text = get_last_message_text(result)
        if not text:
            raise ValueError('Agent returned no message content.')
        return text
    except Exception as e:
        print(f"[offline-fallback] Agent call failed ({type(e).__name__}: {e}).")
        return f"[offline-fallback] {fallback_text}"


def research_node(state: SupervisorState):
    prompt = (
        "Research Larkspur's policy handbook for a new-hire briefing.\n\n"
        f'Question: {state["question"]}\n\n'
        'Return 3 to 5 evidence bullets with page references if possible.'
    )
    notes = safe_agent_invoke(
        research_agent,
        prompt,
        fallback_text=(
            "No live Groq connection, so the Researcher agent couldn't run. With a real "
            "GROQ_API_KEY, this would return evidence bullets pulled from retrieve_evidence "
            "against the handbook chunks indexed above."
        ),
    )
    return {'research_notes': notes}

def write_node(state: SupervisorState):
    prompt = (
        'Write a concise onboarding brief for a new support hire.\n\n'
        f'Question: {state["question"]}\n\n'
        f'Research notes:\n{state.get("research_notes", "")}\n\n'
        'Use headings: Overview, Key Evidence, Why It Matters.'
    )
    draft = safe_agent_invoke(
        writer_agent,
        prompt,
        fallback_text=(
            "No live Groq connection, so the Writer agent couldn't run. With a real "
            "GROQ_API_KEY, this would turn the research notes above into a structured brief."
        ),
    )
    return {'draft': draft}

def qa_node(state: SupervisorState):
    prompt = (
        'Review the draft for unsupported claims and improve clarity.\n\n'
        f'Question: {state["question"]}\n\n'
        f'Research notes:\n{state.get("research_notes", "")}\n\n'
        f'Draft:\n{state.get("draft", "")}\n\n'
        'Return a short QA report followed by a revised final brief.'
    )
    qa_text = safe_agent_invoke(
        qa_agent,
        prompt,
        fallback_text=(
            "No live Groq connection, so the QA agent couldn't run. With a real "
            "GROQ_API_KEY, this would check the draft above against the research notes "
            "and return a QA report plus a revised brief."
        ),
    )
    return {'qa_report': qa_text, 'final_brief': qa_text}

def finish_node(state: SupervisorState):
    final_text = state.get('final_brief') or state.get('draft') or state.get('research_notes', '')
    return {'final_brief': final_text}


## 11) Build the supervisor graph

The graph compiles into a deterministic workflow, and the supervisor decides which specialist runs next.


In [ ]:
builder = StateGraph(SupervisorState)

builder.add_node('supervisor', supervisor_node)
builder.add_node('research', research_node)
builder.add_node('write', write_node)
builder.add_node('qa', qa_node)
builder.add_node('finish', finish_node)

builder.add_edge(START, 'supervisor')
builder.add_conditional_edges(
    'supervisor',
    route_from_supervisor,
    {
        'research': 'research',
        'write': 'write',
        'qa': 'qa',
        'finish': 'finish',
    },
)
builder.add_edge('research', 'supervisor')
builder.add_edge('write', 'supervisor')
builder.add_edge('qa', 'supervisor')
builder.add_edge('finish', END)

graph = builder.compile(checkpointer=InMemorySaver())

print('Supervisor graph compiled.')


## 12) Visualize the graph

The multi-agent docs emphasize that the main agent coordinates the subagents. A simple graph visualization helps show that flow clearly.

The original version of this cell rendered a PNG via `draw_mermaid_png()`, which calls a hosted rendering service (mermaid.ink) to turn the diagram into an image — a real network dependency with nothing to do with Larkspur's own stack, and one more thing that can fail on a machine without outbound access to that specific third-party host. Every other notebook in this module uses the plain-text `draw_mermaid()` output instead, so we do the same here for consistency: same diagram, no external rendering service involved.


In [ ]:
mermaid = graph.get_graph().draw_mermaid()
print(mermaid)


## 13) Run the first request

The agent should go through research, writing, and QA before producing the final brief.


In [ ]:
config = {'configurable': {'thread_id': 'onboarding-brief-1'}}

result = graph.invoke(
    {
        'messages': [],
        'question': 'Create an onboarding briefing for new support hires on our refund and warranty policy.',
        'stage': '',
        'research_notes': '',
        'draft': '',
        'qa_report': '',
        'final_brief': '',
    },
    config=config,
)

result


## 14) Ask a follow-up in the same thread

The same checkpointed thread can be reused for a revised request.


In [ ]:
follow_up = graph.invoke(
    {
        'messages': [],
        'question': 'Now make it shorter and focus on the warranty exclusions.',
        'stage': '',
        'research_notes': '',
        'draft': '',
        'qa_report': '',
        'final_brief': '',
    },
    config=config,
)

follow_up


## 15) What each agent is responsible for

- Researcher: gather evidence from the policy handbook
- Writer: turn evidence into a readable onboarding brief
- QA: identify unsupported claims and revise for clarity
- Supervisor: decide which subagent runs next

That division matches the LangChain multi-agent pattern where specialized agents are coordinated by a main agent.

## Key takeaways

This notebook uses a real, handbook-grounded onboarding-brief workflow and a supervisor graph that coordinates specialized researcher, writer, and QA agents. The design matches LangChain's multi-agent guidance: a main agent coordinates subagents, centralized control stays in one place, and the system is built around context engineering and specialization.

### What's verified where

| Piece | Status |
|---|---|
| Supervisor routing, graph wiring, checkpointing, PDF generation and chunking | Executed and verified in this sandbox. |
| Chroma retrieval | Verified using the `LocalHashEmbeddings` fallback in this sandbox. Retrieval quality will improve with the real `sentence-transformers` model on your machine. |
| Researcher / Writer / QA subagent calls | **Not available in this sandbox** — every run above shows the `[offline-fallback]` path for all three. Confirm the actual generated research notes, draft, and QA report on your own machine with a real `GROQ_API_KEY`. |
| `langgraph-checkpoint-memory` in the original install list | This was never a real published package — `InMemorySaver` ships inside core `langgraph`. Removed from the install cell rather than left to fail with a confusing "no matching distribution" error. |

## References

- Component architecture: https://docs.langchain.com/oss/python/langchain/component-architecture#multi-agent-system
- Multi-agent patterns: https://docs.langchain.com/oss/python/langchain/multi-agent/index#patterns
- Subagents: https://docs.langchain.com/oss/python/langchain/multi-agent/subagents
